## Project initialization

Create a project: a dedicated space where we can manage functions, artifacts and executions.

Use the ``username`` as param to create a personal project in the shared space.

In [ ]:
import digitalhub as dh
import os

project = dh.get_or_create_project(f"{os.environ['USER']}-base")
project

## Scenario 1. More complex example - geodata 

Do some geodata exploration using public data

In [ ]:
%pip install geopandas contextily

In [ ]:
import geopandas 


url = "https://dati.meteotrentino.it/service.asmx/getHumidexGeoJson"
df = geopandas.read_file(url)
df.head()

In [ ]:
import contextily as cx

ax = df.plot()
cx.add_basemap(ax, crs=df.crs)
ax.figure.savefig('foo.pdf')

### 2.1. Create and run function locally

Pick the code from notebook cells and define a function.
Then execute the function in the local env (`local=True`)

In [ ]:
%%writefile "hellogeo.py"

import geopandas 
import contextily as cx

# def geoprocessing():

In [ ]:
func = project.new_function(name="geo-job",
                            kind="python",
                            python_version="PYTHON3_10",
                            code_src="hellogeo.py",
                            handler="geoprocessing"
                           )

In [ ]:
run = func.run("job", wait=True, local_execution=True)

### 2.2. Execute as remote

Execute the function as batch job (`local=False`).
Does it work?

If not, figure out the error and then fix the function definition. Hint: check logs

In [ ]:
run = func.run("job", wait=True, local_execution=False)

### 2.3 Persist the output

Maybe we want to *persist* the result of the work...

* log as artifact
* return the artifact as function *output*

ref https://scc-digitalhub.github.io/sdk-docs/0.14/reference/objects/artifact/crud/#digitalhub.entities.artifact.crud.log_artifact

In [ ]:
%%writefile "hellogeo.py"

import geopandas 
import contextily as cx

# def geoprocessing(project):
    ...
    return project.log_artifact(name="foo.pdf", kind="artifact", source="foo.pdf")


In [ ]:
func = project.new_function(name="geo-job",
                            kind="python",
                            python_version="PYTHON3_10",
                            code_src="hellogeo.py",
                            handler="geoprocessing"
                           )

In [ ]:

run = func.run("job", wait=True, local_execution=False)